# 🛠️ Guide Technique — Projet MaintiDoc

> Documentation d'installation, de l'architecture des bibliothèques et du workflow quotidien du projet MaintiDoc.

## 📑 Table des matières

1. [L'outil d'installation : `uv`](#1-loutil-dinstallation--uv)
2. [Les bibliothèques par étape du projet](#2-les-bibliothèques-par-étape-du-projet)
3. [Workflow quotidien](#3-workflow-quotidien)
4. [Argumentaire MLOps (pour la soutenance)](#4-argumentaire-mlops-pour-la-soutenance)

## 1. L'outil d'installation : `uv`

### 🔄 Changement de paradigme

| Ancienne méthode | Nouvelle méthode |
|---|---|
| `pip install -r requirements.txt` | `uv sync` (via `pyproject.toml` + `uv.lock`) |

### 📂 Fichiers clés

| Fichier / Dossier | Rôle |
|---|---|
| **`.venv/`** | Environnement virtuel isolé créé par `uv`. Toutes les bibliothèques y sont installées sans polluer le PC. |
| **`pyproject.toml`** | Nouvelle « carte d'identité » du projet. Remplace `requirements.txt` et liste les dépendances. |
| **`uv.lock`** | 🔒 **Le plus important.** Fige les versions **exactes** de toutes les librairies pour garantir un comportement identique sur toutes les machines (PC étudiant = PC professeur). |

### ✅ Avantages

- Isolation totale de l'environnement
- Reproductibilité garantie (versions figées)
- Installation ultra-rapide

## 2. Les bibliothèques par étape du projet

### 📄 ÉTAPE 1 — Ingestion & Stockage Vectoriel

| Bibliothèque | Rôle |
|---|---|
| **`pymupdf`** | Ouvrir le PDF (manuel SAN) et extraire le texte, les tableaux et les images. |
| **`pillow` (PIL)** | Manipuler les images extraites du PDF (redimensionnement, préparation pour le modèle Vision). |
| **`sentence-transformers`** | Transformer les textes en **vecteurs** pour permettre une recherche par le sens (et non par mots-clés). |
| **`qdrant-client`** | Connecteur pour la base de données vectorielle **Qdrant** (stockage des textes, images et métadonnées : page, `document_id`, etc.). |

### 🔍 ÉTAPE 2 — Recherche & Reranking

| Bibliothèque | Rôle |
|---|---|
| **`sentence-transformers`** *(réutilisé)* | Fournit également les modèles **CrossEncoder** pour le **Reranking** (reclasser les résultats du plus pertinent au moins pertinent). |

### 🧠 ÉTAPES 3 & 4 — Le Cerveau (Agent) & Le Serveur MCP

| Bibliothèque | Rôle |
|---|---|
| **`torch` (PyTorch)** & **`torchvision`** | Moteur d'IA. La config **CUDA 12.8** permet d'exploiter la carte graphique **RTX 5050** pour des performances élevées. |
| **`transformers`, `accelerate`, `bitsandbytes`** | Charger et faire tourner le LLM et le VLM (Qwen) tout en économisant la mémoire GPU. |
| **`qwen-vl-utils`** | Fonctions dédiées pour préparer images et texte spécifiquement pour **Qwen**. |
| **`langchain`, `langchain-community`** | Framework pour créer les **outils** que l'IA peut utiliser. |
| **`langgraph`** | ⚙️ **Package central de l'Étape 3.** Crée le **workflow (graphe)** de l'agent : gestion des états, des boucles et de l'**interruption pour validation humaine**. |
| **`fastapi` & `uvicorn`** | Serveur web : FastAPI crée les routes, Uvicorn les rend accessibles (`mcp_server.py`). |
| **`httpx`** | Requêtes HTTP. L'agent LangGraph l'utilise pour appeler le serveur MCP. |
| **`pydantic`** | Validation des données (ex. : garantir un JSON strict pour les tickets). |

### 🖥️ Interface & Sécurité

| Bibliothèque | Rôle |
|---|---|
| **`streamlit`** | Interface web visuelle (questions, bouton « Approuver »). |
| **`python-dotenv`** | Charger les secrets (ex. `APPROVAL_SECRET`) depuis un fichier `.env` **jamais partagé**. |

### 🧪 Évaluation & Qualité (pour le prof)

| Bibliothèque | Rôle |
|---|---|
| **`pytest`** | Lancer les tests unitaires (ex. `test_mcp.py`). |
| **`mlflow`** | Suivre les expériences : meilleur modèle, temps de réponse… (Étape 6). |
| **`jupyterlab` & `ipykernel`** | Coder dans des Notebooks (idéal pour tester l'ingestion PDF étape par étape). |

## 3. Workflow quotidien

> ⚠️ **Règle d'or :** ne **jamais** taper `python mon_script.py`.  
> Toujours utiliser **`uv run`** pour que le script s'exécute dans le bon environnement virtuel.

### 1️⃣ Vérifier que le GPU est prêt

```powershell
uv run python scripts\check_env.py
```

✅ Si CUDA est disponible → prêt pour l'IA.

### 2️⃣ Ouvrir un Notebook Jupyter (Étape 1)

```powershell
uv run jupyter lab
```

Créer un fichier `notebooks/01_ingestion_test.ipynb` pour tester `pymupdf` sur le PDF et vérifier l'extraction du texte.

### 3️⃣ Lancer le serveur MCP (Étape 4)

```powershell
uv run uvicorn mcp_server:app --port 8001
```

### 4️⃣ Lancer les tests de sécurité

```powershell
uv run pytest
```

## 4. Argumentaire MLOps (pour la soutenance)

> *« Monsieur, pour garantir la **reproductibilité** de notre projet (MLOps — Étape 7), nous n'avons pas utilisé `pip`. Nous avons utilisé **`uv`** pour gérer un environnement virtuel isolé basé sur un fichier **`pyproject.toml`** et **`uv.lock`**. Cela garantit que le projet tournera exactement de la même manière sur n'importe quelle machine. »*

### 📌 Récapitulatif des commandes essentielles

| Action | Commande |
|---|---|
| Vérifier l'environnement GPU | `uv run python scripts\check_env.py` |
| Ouvrir Jupyter Lab | `uv run jupyter lab` |
| Lancer le serveur MCP | `uv run uvicorn mcp_server:app --port 8001` |
| Lancer les tests | `uv run pytest` |

*Document généré pour le projet MaintiDoc — Étape 7 (MLOps / Reproductibilité).*